In [1]:
import os
import pandas as pd
import numpy as np
import time
from supabase import create_client
from dotenv import load_dotenv

# 1. 환경 변수 및 DB 세팅
load_dotenv()
SUPABASE_URL = os.environ.get("SUPABASE_URL")
SUPABASE_KEY = os.environ.get("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

CSV_PATH = "./nikke_arts/metadata.csv"

def migrate_csv_to_supabase(csv_path):
    print(f"🚀 [마이그레이션 시작] {csv_path} 파일을 읽어옵니다...")
    
    # 데이터 로드
    df = pd.read_csv(csv_path)
    total_records = len(df)
    
    # 💡 [핵심] 필요 없는 컬럼(filename) 삭제 및 UUID 문자열 캐스팅
    if 'filename' in df.columns:
        df = df.drop(columns=['filename'])
    df['post_uuid'] = df['post_uuid'].astype(str)
    
    # 💡 [핵심] Pandas의 NaN, NaT 등을 DB가 인식할 수 있는 None으로 완벽 치환
    df = df.replace({np.nan: None})
    
    # DataFrame을 Dictionary 리스트로 변환
    records = df.to_dict('records')
    
    # 💡 [핵심] 대용량 데이터를 한 번에 보내면 서버가 터지므로 청크(Chunk) 단위로 쪼개서 전송
    chunk_size = 500
    successful_inserts = 0
    
    print(f"총 {total_records}개의 데이터를 {chunk_size}개씩 쪼개서 업로드합니다.")
    print("-" * 50)
    
    for i in range(0, total_records, chunk_size):
        chunk = records[i : i + chunk_size]
        
        try:
            # upsert를 사용하여 만약 이미 들어간 데이터가 있다면 덮어쓰기(안전장치)
            res = supabase.table("nikke_arts").upsert(chunk).execute()
            successful_inserts += len(chunk)
            print(f"✅ [{i} ~ {i + len(chunk) - 1}] 배치 업로드 성공! (누적: {successful_inserts}/{total_records})")
            
            # 서버 과부하 방지
            time.sleep(0.5)
            
        except Exception as e:
            print(f"❌ 에러 발생 구간 [{i} ~ {i + len(chunk) - 1}]: {e}")
            print("데이터 타입 불일치 또는 결측치 치환 문제가 원인일 수 있습니다.")
            break
            
    print("-" * 50)
    print(f"🏁 [마이그레이션 완료] 총 {successful_inserts}건의 데이터가 Supabase에 적재되었습니다.")

# 실행!
migrate_csv_to_supabase(CSV_PATH)

SupabaseException: supabase_url is required